# Gemini 3.8 Flash: gợi ý nhãn cho các query còn lại

Notebook local này đọc `review/query_candidates_40.xlsx` đang có và hai JPEG trong `review/query_images/`. Các dòng đã có nhãn người được giữ nguyên. Kết quả VLM nằm trong thư mục `review/vlm_runs/<lượt>/`, gồm JSONL kiểm toán, CSV và Excel bản nháp có ảnh. Không sửa workbook gốc hoặc CSV ứng viên.

VLM hỗ trợ sàng lọc; `review_status=vlm_draft` **không** phải nhãn `approved`. API key được nhập ẩn hoặc qua biến môi trường, không lưu trong notebook.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, importlib.util, json, os, subprocess, sys, time
from getpass import getpass

cwd = Path.cwd().resolve()
search = [cwd, *cwd.parents, cwd/'duong', cwd/'graphml-project3/duong']
DUONG = next((p for p in search if (p/'xai/vlm_review.py').is_file()), None)
assert DUONG is not None, 'Open notebook from the graphml workspace.'
sys.path.insert(0, str(DUONG))
from xai.vlm_review import load_queue, image_paths, completed_records, matching_record, export_draft

REVIEW = DUONG/'review'
INPUT_WORKBOOK = REVIEW/'query_candidates_40.xlsx'  # workbook bạn đã điền một phần
REGISTRY = REVIEW/'queries_relational_candidates.csv'
IMAGE_DIR = REVIEW/'query_images'
MODEL_ID = 'gemini-3.8-flash'
MAX_ROWS = None  # đặt 2 để thử trước; đổi lại None rồi chạy tiếp
registry, reviews, human_keys, queue = load_queue(REGISTRY, INPUT_WORKBOOK, IMAGE_DIR)
print('Đã có nhãn người:', len(human_keys), '| VLM cần chấm:', len(queue))
RUN_DIR = None  # để None tạo lượt mới; đặt đường dẫn lượt cũ để tiếp tục sau khi khởi động lại
RUN_DIR = Path(RUN_DIR) if RUN_DIR else REVIEW/'vlm_runs'/('gemini38_'+datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S'))
RUN_DIR.mkdir(parents=True, exist_ok=True)
LOG = RUN_DIR/'assessments.jsonl'
print('Thư mục kết quả:', RUN_DIR)


## 1. Kết nối API và rubric

Ảnh A là ảnh dương đề xuất; ảnh B là ảnh âm đề xuất. Mỗi yêu cầu có query, hai ảnh và schema JSON cố định. Nhãn `hard_negative_same_objects` đòi hỏi **đủ mọi đối tượng thiết yếu**, còn `relation_differs` chỉ đúng khi có cùng các đối tượng nhưng quan hệ/vai trò khác.

In [ ]:
if importlib.util.find_spec('google') is None or importlib.util.find_spec('google.genai') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'google-genai>=1.0'], check=True)
from google import genai
from google.genai import types
from pydantic import BaseModel
from typing import Literal

Status = Literal['supported', 'contradicted', 'uncertain']
class Assessment(BaseModel):
    verified_group: Literal['direct', 'relational', 'uncertain']
    claim_subject: str
    claim_predicate: str
    claim_object: str
    gt_relevance: Status
    retrieved_relevance: Status
    hard_negative_same_objects: Status
    relation_differs: Status
    visual_reason: str
    image_a_observation: str
    image_b_observation: str
    uncertainty_notes: str
    confidence: Literal['low', 'medium', 'high']

RUBRIC = '''Bạn là người chấm độc lập cho text-to-image retrieval. Query là claim cần kiểm tra, không phải sự thật về ảnh.
Ảnh A là ảnh dương do dataset đề xuất. Ảnh B là ảnh âm khó đề xuất. Quan sát từng ảnh riêng rồi so sánh.
- gt_relevance / retrieved_relevance: supported chỉ khi ảnh hỗ trợ TOÀN BỘ câu; contradicted nếu có bằng chứng rõ chống lại câu; uncertain nếu chi tiết không nhận biết được.
- hard_negative_same_objects: supported chỉ khi CẢ HAI ảnh có ĐỦ mọi loại đối tượng thiết yếu của claim. Cùng một đối tượng là chưa đủ.
- relation_differs: supported chỉ khi các đối tượng thiết yếu có ở cả hai ảnh và quan hệ/vai trò khác nhau. Nếu thiếu đối tượng, ghi uncertain. Nếu chỉ khác loại món, màu hay thuộc tính, ghi contradicted.
- verified_group: direct nếu claim kiểm tra được qua một quan hệ/hành động nhìn thấy trực tiếp; relational nếu cần nối nhiều quan hệ hoặc suy vai trò gián tiếp; uncertain nếu không rõ.
- Chọn bộ ba subject–predicate–object chính phân biệt ảnh. Với câu nhiều claim, ghi các điều kiện còn lại trong visual_reason.
- Không suy ra tên món, chất liệu, ý định hoặc trạng thái không thấy rõ chỉ từ màu sắc. Ghi uncertain nếu thiếu bằng chứng.
- visual_reason nêu rõ ảnh A/B có gì, vì sao gán nhãn và giới hạn quan sát, bằng tiếng Việt.
Không tự gán review_status=approved; chỉ người chấm quyết định.'''
key = os.environ.get('GEMINI_API_KEY') or getpass('GEMINI_API_KEY (ẩn): ')
assert key, 'Cần API key để gọi Gemini.'
client = genai.Client(api_key=key)
del key
print('Sẵn sàng:', MODEL_ID)


## 2. Chấm tuần tự và lưu sau từng dòng

Đặt `MAX_ROWS=2` để thử. Sau đó đặt `None` và chạy lại cell này trong cùng kernel; JSONL sẽ giúp bỏ qua các dòng đã chấm thành công khi model và hash hai ảnh còn khớp. Nếu khởi động lại kernel, đặt `RUN_DIR` trong cell đầu thành thư mục lượt cũ. Lỗi được ghi lại và có thể thử lại.

In [ ]:
def assess(row):
    gt_path, neg_path = image_paths(IMAGE_DIR, row)
    gt_bytes, neg_bytes = gt_path.read_bytes(), neg_path.read_bytes()
    prompt = RUBRIC + f"\nDataset: {row['dataset']}; query_id: {row['query_id']}; Query: {row['query']}\nẢnh A:"
    parts = [types.Part.from_text(text=prompt),
             types.Part.from_bytes(data=gt_bytes, mime_type='image/jpeg'),
             types.Part.from_text(text='Ảnh B:'),
             types.Part.from_bytes(data=neg_bytes, mime_type='image/jpeg')]
    for attempt in range(4):
        try:
            response = client.models.generate_content(
                model=MODEL_ID,
                contents=[types.Content(role='user', parts=parts)],
                config=types.GenerateContentConfig(
                    temperature=0, response_mime_type='application/json',
                    response_schema=Assessment))
            result = Assessment.model_validate_json(response.text).model_dump()
            return {'dataset': row['dataset'], 'query_id': row['query_id'],
                    'status': 'ok', 'model': MODEL_ID, 'result': result,
                    'query_sha256': hashlib.sha256(row['query'].encode('utf-8')).hexdigest(),
                    'gt_sha256': hashlib.sha256(gt_bytes).hexdigest(),
                    'negative_sha256': hashlib.sha256(neg_bytes).hexdigest(),
                    'timestamp_utc': datetime.now(timezone.utc).isoformat(),
                    'total_tokens': getattr(response.usage_metadata, 'total_token_count', None)}
        except Exception as exc:
            code = getattr(exc, 'code', None) or getattr(exc, 'status_code', None)
            if attempt < 3 and (code in (429, 500, 502, 503, 504) or '429' in str(exc)):
                time.sleep(min(30, 2 ** (attempt + 1)))
                continue
            return {'dataset': row['dataset'], 'query_id': row['query_id'],
                    'status': 'error', 'model': MODEL_ID, 'error': str(exc),
                    'timestamp_utc': datetime.now(timezone.utc).isoformat()}

done = completed_records(LOG)
remaining = [row for row in queue if not matching_record(
    done.get((row['dataset'], row['query_id'])), row, IMAGE_DIR, MODEL_ID)]
if MAX_ROWS is not None: remaining = remaining[:MAX_ROWS]
print('Gọi model cho', len(remaining), 'dòng')
for index, row in enumerate(remaining, 1):
    record = assess(row)
    with LOG.open('a', encoding='utf-8') as stream:
        stream.write(json.dumps(record, ensure_ascii=False) + '\n')
    if record['status'] == 'ok':
        done[(row['dataset'], row['query_id'])] = record
    print(index, '/', len(remaining), row['dataset'], row['query_id'], record['status'])
print('Bản nháp thành công:', len(done), '/', len(queue))


## 3. Xuất CSV và Excel bản nháp

Cell này đọc JSONL, giữ nguyên các dòng đã có nhãn người và điền nhãn VLM cho những dòng còn lại. File Excel đầu ra nhúng cả hai ảnh mỗi query. Bạn kiểm tra, sửa và phê duyệt thủ công trước khi dùng kết quả.

In [ ]:
csv_path, xlsx_path, counts = export_draft(
    registry, reviews, human_keys, completed_records(LOG), IMAGE_DIR, RUN_DIR, MODEL_ID)
print('CSV:', csv_path)
print('Excel:', xlsx_path)
print('Số dòng:', counts)
